# Crash Course — 임상 예측·조기진단 모델

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [crash-course/clinical-prediction.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/crash-course/clinical-prediction.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 최소 실행 예시

### 예시 1 — 같은 도구, 다른 유병률에서의 PPV

In [ ]:
import numpy as np
import pandas as pd

sens, spec = 0.85, 0.90          # 가상의 선별 도구: 민감도 85%, 특이도 90%
rows = []
for setting, prev in [("환자-대조군 연구(1:1)", 0.50),
                      ("고위험 의뢰 클리닉", 0.10),
                      ("일반 인구 선별", 0.005)]:
    n = 10000
    diseased = n * prev
    tp = sens * diseased
    fp = (1 - spec) * (n - diseased)
    fn = diseased - tp
    tn = (n - diseased) - fp
    rows.append({"상황": setting, "유병률": prev, "양성 판정": round(tp + fp),
                 "진양성": round(tp), "위양성": round(fp),
                 "PPV": round(tp / (tp + fp), 3), "NPV": round(tn / (tn + fn), 4)})
print(pd.DataFrame(rows).to_string(index=False))

### 예시 2 — 외부 기관에서의 보정과 순 이득, 그리고 재보정

In [ ]:
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

rng = np.random.default_rng(0)
def make(n, base=-1.6):
    X = rng.normal(size=(n, 2))
    y = rng.random(n) < 1 / (1 + np.exp(-(base + X @ [1.0, 0.5])))
    return X, y.astype(int)

X_dev, y_dev = make(1000)
X_ext, y_ext = make(1000, base=-2.6)          # 외부 기관: 기저 위험이 더 낮다
p = LogisticRegression().fit(X_dev, y_dev).predict_proba(X_ext)[:, 1]
lp = np.log(p / (1 - p))

citl = sm.GLM(y_ext, np.ones((len(p), 1)), family=sm.families.Binomial(), offset=lp).fit().params[0]
slope = sm.Logit(y_ext, sm.add_constant(lp)).fit(disp=0).params[1]
print(f"AUC {roc_auc_score(y_ext, p):.3f} | Brier {brier_score_loss(y_ext, p):.3f} | 관찰 {y_ext.mean():.3f} 예측 {p.mean():.3f}"
      f" | 보정 절편 {citl:+.2f} 기울기 {slope:.2f}")

def net_benefit(y, p, pt):
    pos = p >= pt
    return (np.sum(pos & (y == 1)) - np.sum(pos & (y == 0)) * pt / (1 - pt)) / len(y)
p_recal = 1 / (1 + np.exp(-(lp + citl)))       # 절편만 재보정(원래는 별도 자료로 추정해야 한다)
for pt in [0.1, 0.2, 0.3]:
    print(f"pt {pt:.1f} | 원래 모델 {net_benefit(y_ext, p, pt):+.3f} | 재보정 {net_benefit(y_ext, p_recal, pt):+.3f}"
          f" | 모두 양성 {y_ext.mean() - (1 - y_ext.mean()) * pt / (1 - pt):+.3f}")